## LoRA Fine-Tune Optimization_Summarize Dialogue_FLAN-T5_247M

## FLAN-T5 247M base
 FLAN‑T5 is an encoder–decoder (seq2seq) model that reads the full input before generating output.
 It uses a text‑to‑text framework, turning every task into input text → output text.
 Its key innovation is instruction fine‑tuning (FLAN) on many natural language tasks.
 This makes it strong at summarization, translation, classification, Q&A and structured generation.
 Compared to GPT‑style models, it is often more controlled and task‑focused.

In [ ]:
import os
os.environ["TORCH_CPP_LOG_LEVEL"] = "ERROR"

In [ ]:
# Cell 1: L40S 46GB STACK - NO bitsandbytes (for Optuna)
# !pip uninstall torch torchvision torchaudio bitsandbytes transformers accelerate peft datasets evaluate -y -q
# First upgrade pip
! pip install --upgrade pip
# PyTorch CUDA 12.1 (L40S certified)
!pip install torch==2.2.1+cu121 torchvision==0.17.1+cu121 torchaudio==2.2.1+cu121 --index-url https://download.pytorch.org/whl/cu121

# HF Stack (your exact versions - NO bitsandbytes)
!pip install transformers==4.40.0 datasets==2.19.0 peft==0.10.0 evaluate==0.4.2 rouge_score==0.1.2 accelerate==0.30.0

# Optuna + Utils (NO bitsandbytes)
!pip install optuna==3.6.1 optuna-integration sentencepiece tqdm

# Data science (pinned)
!pip install pandas==2.2.2 numpy==1.26.4 scipy==1.13.1 scikit-learn==1.3.2 matplotlib==3.8.2

print("✅ L40S 46GB STACK - NO bitsandbytes ERROR")
print("🔄 RESTART KERNEL → Test → Optuna")

Looking in indexes: https://download.pytorch.org/whl/cu121
✅ L40S 46GB STACK - NO bitsandbytes ERROR
🔄 RESTART KERNEL → Test → Optuna


In [2]:
#!pip uninstall torch torchvision torchaudio transformers accelerate peft datasets evaluate -y -q

In [ ]:
#
import torch
print(torch.cuda.is_available())

False


In [ ]:
# Imports \& Global Config
import os, time, json, warnings
warnings.filterwarnings("ignore")

import torch
import numpy as np
import pandas as pd
import optuna
from optuna.samplers import TPESampler

from datasets import load_dataset
from transformers import (
    AutoModelForSeq2SeqLM, AutoTokenizer,
    TrainingArguments, Trainer,
    EarlyStoppingCallback,
)
from peft import LoraConfig, get_peft_model, TaskType, PeftModel
import evaluate

print(f"PyTorch : {torch.__version__}")
print(f"CUDA    : {torch.cuda.is_available()} | "
      f"Device  : {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'}")


W0412 23:00:08.436000 16304 Lib\site-packages\torch\distributed\elastic\multiprocessing\redirects.py:29] NOTE: Redirects are currently not supported in Windows or MacOs.


PyTorch : 2.11.0+cpu
CUDA    : False | Device  : CPU


In [ ]:
## 📋 Cell 3 — Output Directory (Lightning.AI Local Storage)

# **Replaces AWS S3** — all weights saved locally. On Lightning.AI, change `BASE_DIR` to `/teamspace/studios/this_studio/output` for persistent cross-session storage.


BASE_DIR      = "./output"
FULL_FT_DIR   = os.path.join(BASE_DIR, "full_finetune_model")
LORA_BEST_DIR = os.path.join(BASE_DIR, "lora_best_model")
OPTUNA_DIR    = os.path.join(BASE_DIR, "optuna_trials")
LOGS_DIR      = os.path.join(BASE_DIR, "logs")

for d in [FULL_FT_DIR, LORA_BEST_DIR, OPTUNA_DIR, LOGS_DIR]:
    os.makedirs(d, exist_ok=True)

print("📁 Output folders ready:")
for d in [BASE_DIR, FULL_FT_DIR, LORA_BEST_DIR, OPTUNA_DIR, LOGS_DIR]:
    print(f"   {d}")
print("\n✅ No AWS S3 needed — Lightning.AI local storage active.")


📁 Output folders ready:
   ./output
   ./output\full_finetune_model
   ./output\lora_best_model
   ./output\optuna_trials
   ./output\logs

✅ No AWS S3 needed — Lightning.AI local storage active.


<a name='1.2'></a>
### Load Dataset and LLM

You are going to continue experimenting with the [DialogSum](https://huggingface.co/datasets/knkarthick/dialogsum) Hugging Face dataset. It contains 10,000+ dialogues with the corresponding manually labeled summaries and topics. 

Load the pre-trained [FLAN-T5 model](https://huggingface.co/docs/transformers/model_doc/flan-t5) and its tokenizer directly from HuggingFace. Notice that you will be using the [small version](https://huggingface.co/google/flan-t5-base) of FLAN-T5. Setting `torch_dtype=torch.bfloat16` specifies the memory type to be used by this model.


In [ ]:

## 📋 Cell 4 — Load Dataset, Tokenizer \& Base Model


DATASET_NAME = "knkarthick/dialogsum"
MODEL_NAME   = "google/flan-t5-base"
DEVICE       = "cuda" if torch.cuda.is_available() else "cpu"

dataset = load_dataset(DATASET_NAME)
print(dataset)

Generating test split: 100%|██████████| 1500/1500 [00:00<00:00, 75002.75 examples/s]

DatasetDict({
    train: Dataset({
        features: ['id', 'dialogue', 'summary', 'topic'],
        num_rows: 12460
    })
    validation: Dataset({
        features: ['id', 'dialogue', 'summary', 'topic'],
        num_rows: 500
    })
    test: Dataset({
        features: ['id', 'dialogue', 'summary', 'topic'],
        num_rows: 1500
    })
})


In [7]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize_function(example):
    start_prompt = "Summarize the following conversation.\n\n"
    end_prompt   = "\n\nSummary: "
    prompts = [start_prompt + d + end_prompt for d in example["dialogue"]]
    inp    = tokenizer(prompts, padding="max_length", truncation=True,
                       max_length=512, return_tensors="pt")
    labels = tokenizer(example["summary"], padding="max_length", truncation=True,
                       max_length=128, return_tensors="pt")
    example["input_ids"]      = inp.input_ids
    example["attention_mask"] = inp.attention_mask
    example["labels"]         = labels.input_ids
    return example

tokenized_datasets = dataset.map(
    tokenize_function, batched=True,
    remove_columns=["id", "topic", "dialogue", "summary"]
)
print(tokenized_datasets)

tokenizer_config.json: 0.00B [00:00, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

Map:   0%|          | 0/12460 [00:00<?, ? examples/s]

Map:   0%|          | 0/500 [00:00<?, ? examples/s]

Map:   0%|          | 0/1500 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['input_ids', 'attention_mask', 'labels'],
        num_rows: 12460
    })
    validation: Dataset({
        features: ['input_ids', 'attention_mask', 'labels'],
        num_rows: 500
    })
    test: Dataset({
        features: ['input_ids', 'attention_mask', 'labels'],
        num_rows: 1500
    })
})


In [8]:
## 📋 Cell 5 — Helper Utilities
## Error
rouge_metric = evaluate.load("rouge")

def count_params(model):
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    total     = sum(p.numel() for p in model.parameters())
    return trainable, total, 100 * trainable / total

def generate_summary(model, prompt_text, max_new_tokens=128):
    inputs = tokenizer(prompt_text, return_tensors="pt",
                       truncation=True, max_length=512).to(DEVICE)
    model.eval()
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=max_new_tokens,
                              num_beams=4, early_stopping=True)
    return tokenizer.decode(out[0], skip_special_tokens=True)

def compute_rouge(model, tokenizer, split, n_samples=50, max_new_tokens=128):
    model.eval()
    preds, refs = [], []
    idxs = np.random.choice(len(split), min(n_samples, len(split)), replace=False)
    for idx in idxs:
        inp = torch.tensor(split[int(idx)]["input_ids"]).unsqueeze(0).to(DEVICE)
        with torch.no_grad():
            out = model.generate(inp, max_new_tokens=max_new_tokens,
                                  num_beams=4, early_stopping=True)
        preds.append(tokenizer.decode(out[0], skip_special_tokens=True))
        refs.append(tokenizer.decode(split[int(idx)]["labels"], skip_special_tokens=True))
    scores = rouge_metric.compute(predictions=preds, references=refs, use_stemmer=True)
    return {k: round(v * 100, 4) for k, v in scores.items()}

print("✅ Helper functions ready.")


✅ Helper functions ready.


In [9]:
def compute_rouge(model, tokenizer, split, n_samples=50, max_new_tokens=128):
    model.eval()
    preds, refs = [], []
    idxs = np.random.choice(len(split), min(n_samples, len(split)), replace=False)
    
    for idx in idxs:
        item = split[int(idx)]
        
        # Extract input_ids AND attention_mask
        input_ids = torch.tensor(item["input_ids"], dtype=torch.long).unsqueeze(0).to(DEVICE)
        attention_mask = torch.tensor(item["attention_mask"], dtype=torch.long).unsqueeze(0).to(DEVICE)
        
        with torch.no_grad():
            # ✅ FIXED: Keyword args for PEFT models
            out = model.generate(
                input_ids=input_ids,           # Keyword!
                attention_mask=attention_mask,  # Keyword! 
                max_new_tokens=max_new_tokens,
                num_beams=4,
                early_stopping=True,
                pad_token_id=tokenizer.pad_token_id  # Required for T5
            )
        
        preds.append(tokenizer.decode(out[0], skip_special_tokens=True))
        refs.append(tokenizer.decode(item["labels"], skip_special_tokens=True))
    
    scores = rouge_metric.compute(predictions=preds, references=refs, use_stemmer=True)
    return {k: round(v * 100, 4) for k, v in scores.items()}

In [10]:
def generate_summary(model, prompt_text, max_new_tokens=128):
    inputs = tokenizer(prompt_text, return_tensors="pt",
                      truncation=True, max_length=512).to(DEVICE)
    model.eval()
    with torch.no_grad():
        # ✅ Already correct (uses **inputs)
        out = model.generate(**inputs, max_new_tokens=max_new_tokens,
                            num_beams=4, early_stopping=True,
                            pad_token_id=tokenizer.pad_token_id)
    return tokenizer.decode(out[0], skip_special_tokens=True)

In [11]:
## 📋 Cell 6 — Zero-Shot Baseline (Original FLAN-T5)
original_model = AutoModelForSeq2SeqLM.from_pretrained(
    MODEL_NAME, torch_dtype=torch.bfloat16
).to(DEVICE)

trainable, total, pct = count_params(original_model)
print(f"Total params     : {total:,}")
print(f"Trainable params : {trainable:,}  ({pct:.2f}%)")

# Quick zero-shot test
index    = 200
dialogue = dataset["test"][index]["dialogue"]
summary  = dataset["test"][index]["summary"]
prompt   = f"Summarize the following conversation.\n\n{dialogue}\n\nSummary: "
out      = generate_summary(original_model, prompt)

print("─" * 80)
print(f"HUMAN SUMMARY  : {summary}")
print("─" * 80)
print(f"ZERO-SHOT MODEL: {out}")

# Zero-shot ROUGE baseline
zeroshot_rouge = compute_rouge(original_model, tokenizer, tokenized_datasets["test"])
print("Zero-Shot ROUGE:", zeroshot_rouge)


config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/990M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

Total params     : 247,577,856
Trainable params : 247,577,856  (100.00%)
────────────────────────────────────────────────────────────────────────────────
HUMAN SUMMARY  : #Person1# teaches #Person2# how to upgrade software and hardware in #Person2#'s system.
────────────────────────────────────────────────────────────────────────────────
ZERO-SHOT MODEL: #Person1#: Have you considered upgrading your system?
Zero-Shot ROUGE: {'rouge1': 21.3912, 'rouge2': 6.0805, 'rougeL': 17.1907, 'rougeLsum': 17.2401}


In [12]:
## 📋 Cell 7 — Full Fine-Tuning (Reference Baseline)

# Trains all 247M params as the ROUGE upper-bound reference.


full_ft_model = AutoModelForSeq2SeqLM.from_pretrained(
    MODEL_NAME, torch_dtype=torch.bfloat16
).to(DEVICE)

full_ft_args = TrainingArguments(
    output_dir=FULL_FT_DIR,
    num_train_epochs=20,
    per_device_train_batch_size=4,        # ↓ Reduced for memory on 46 GPUs
    per_device_eval_batch_size=4,         # ↓ Reduced for memory on 46 GPUs
    gradient_accumulation_steps=4,         # ↑ Effective batch=4*4*46=..
    learning_rate=5e-4,                   # ↑ Higher LR for short training
    weight_decay=0.01,
    warmup_ratio=0.03,                    # ↓ Faster warmup for 1hr
    lr_scheduler_type="cosine",
    logging_steps=10,                     # ↓ More frequent logging
    evaluation_strategy="steps",
    eval_steps=500,                       # ↓ Eval every ~3-5min on 46 GPUs
    save_strategy="steps",
    save_steps=1000,                      # ↓ Save every ~7-10min
    save_total_limit=2,                   # ↓ Only keep 2 checkpoints
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    #fp16=torch.cuda.is_available(),
    #bf16=torch.cuda.is_available() and torch.cuda.get_device_capability()[0] >= 8,  # A100/H100
    fp16=False,
    bf16=torch.cuda.is_available(),
    dataloader_num_workers=0,             # ↓ Critical for multi-GPU stability
    report_to="none",
    dataloader_pin_memory=False,          # ↓ Memory stability on 46 GPUs
    remove_unused_columns=False,          # ↓ Avoid dataset processing issues
)

full_ft_trainer = Trainer(
    model=full_ft_model,
    args=full_ft_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["validation"],
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],
)
print("🔄 Starting full fine-tuning...")
full_ft_trainer.train()
full_ft_model.save_pretrained(FULL_FT_DIR)
tokenizer.save_pretrained(FULL_FT_DIR)
print(f"✅ Saved → {FULL_FT_DIR}")
full_ft_rouge = compute_rouge(full_ft_model, tokenizer, tokenized_datasets["test"])
print("Full Fine-Tune ROUGE:", full_ft_rouge)


🔄 Starting full fine-tuning...


Step,Training Loss,Validation Loss
500,0.367700,0.333490
1000,0.348500,0.316623
1500,0.328100,0.314909
2000,0.301200,0.311669
2500,0.280100,0.317889
3000,0.289200,0.313499


There were missing keys in the checkpoint model loaded: ['encoder.embed_tokens.weight', 'decoder.embed_tokens.weight'].


✅ Saved → ./output/full_finetune_model
Full Fine-Tune ROUGE: {'rouge1': 48.9014, 'rouge2': 22.7941, 'rougeL': 40.1368, 'rougeLsum': 40.2014}


In [17]:
## 📋 Cell 8 — LoRA + Optuna Hyperparameter Optimization

### Search Space
# ── Optuna objective: maximise ROUGE-1 ───────────────────────────────────────
LORA_TARGET_MODULES = ["q", "v"]   # FLAN-T5 attention projections
OPTUNA_N_TRIALS     = 10          # Increase to 30-50 for thorough A40 search

def lora_objective(trial: optuna.Trial) -> float:
    r = trial.suggest_categorical("r", [128])           
    lora_alpha = trial.suggest_int("lora_alpha", 64, 128, step=32)  
    lora_dropout = trial.suggest_float("lora_dropout", 0.03, 0.05)     
    epochs = trial.suggest_int("num_train_epochs", 3, 5)       
    lr = trial.suggest_float("learning_rate", 5e-4, 7e-4, log=True)  
    batch_size = trial.suggest_categorical("batch_size", [16, 24, 32])  # 46GB safe
    #warmup_steps = trial.suggest_int("warmup_steps", 15, 30)
    warmup_ratio = trial.suggest_float("warmup_ratio", 0.1, 0.2)
    
    target_modules = ["q", "v", "k", "o", "gate_proj", "up_proj", "down_proj"]

    # Fresh base model per trial — avoids gradient contamination
    base = AutoModelForSeq2SeqLM.from_pretrained(
        MODEL_NAME, torch_dtype=torch.bfloat16
    ).to(DEVICE)

    
    cfg = LoraConfig(
    task_type=TaskType.SEQ_2_SEQ_LM,
    r=r, 
    lora_alpha=lora_alpha, 
    lora_dropout=lora_dropout,
    target_modules=target_modules,  # All linear layers!
    bias="none",
    use_rslora=True,            # ↑ Rank-stabilized LoRA (SOTA)
    init_lora_weights="loftq",  # Better initialization
)


    peft_model = get_peft_model(base, cfg)

    t_args = TrainingArguments(
        output_dir=os.path.join(OPTUNA_DIR, f"trial_{trial.number}"),
        num_train_epochs=epochs,
        max_steps=80,           # Hard cap per trial on A40 (2hr budget)
        gradient_accumulation_steps=2, # Effective BS scales better
        per_device_train_batch_size=batch_size,
        per_device_eval_batch_size=batch_size,
        learning_rate=lr,
        warmup_ratio=warmup_ratio,
        weight_decay=0.01,
        lr_scheduler_type="cosine",
        evaluation_strategy="steps",
        eval_steps=100,
        save_strategy="no",
        logging_steps=25,
        fp16=False,
        bf16=torch.cuda.is_bf16_supported(),
        dataloader_num_workers=4,
        report_to="none",
        load_best_model_at_end=False,
        
    )
    trainer = Trainer(
        model=peft_model, args=t_args,
        train_dataset=tokenized_datasets["train"],
        eval_dataset=tokenized_datasets["validation"],
    )
    trainer.train()

    scores = compute_rouge(peft_model, tokenizer,
                           tokenized_datasets["validation"], n_samples=30)
    rouge1 = scores["rouge1"]
    print(f"Trial {trial.number:>3} | r={r:>2} α={lora_alpha} "
          f"drop={lora_dropout:.2f} lr={lr:.2e} ep={epochs} "
          f"bs={batch_size} | ROUGE-1={rouge1:.4f}")

    del peft_model, base, trainer
    torch.cuda.empty_cache()
    return rouge1

print("✅ Optuna objective defined.")



# ── Run Optuna study ──────────────────────────────────────────────────────────
import os
os.environ["TOKENIZERS_PARALLELISM"] = "false"  # ← Kills the warning

import torch
from transformers import TrainingArguments, Trainer
# ... rest of imports
study = optuna.create_study(
    direction="maximize",
    sampler=TPESampler(seed=42),
    study_name="lora_rouge1_opt",
)
print(f"🔍 Starting Optuna HPO: {OPTUNA_N_TRIALS} trials on Lightning.AI A40...")
study.optimize(lora_objective, n_trials=OPTUNA_N_TRIALS, show_progress_bar=True)

print("\n" + "═" * 60)
print("🏆 Best Trial:")
best = study.best_trial
print(f"   ROUGE-1 : {best.value:.4f}")
for k, v in best.params.items():
    print(f"   {k:<22}: {v}")

study.trials_dataframe().to_csv(
    os.path.join(OPTUNA_DIR, "optuna_trials.csv"), index=False)
print(f"\n📊 All trial results saved → {OPTUNA_DIR}/optuna_trials.csv")




[I 2026-04-12 07:42:21,049] A new study created in memory with name: lora_rouge1_opt


✅ Optuna objective defined.
🔍 Starting Optuna HPO: 10 trials on Lightning.AI A40...


  0%|          | 0/10 [00:00<?, ?it/s]

max_steps is given, it will override any value given in num_train_epochs


Step,Training Loss,Validation Loss


...

Step,Training Loss,Validation Loss


Trial   8 | r=128 α=64 drop=0.04 lr=6.55e-04 ep=3 bs=24 | ROUGE-1=41.4780
[I 2026-04-12 07:57:33,883] Trial 8 finished with value: 41.478 and parameters: {'r': 128, 'lora_alpha': 64, 'lora_dropout': 0.04085392166316497, 'num_train_epochs': 3, 'learning_rate': 0.0006549278721699911, 'batch_size': 24, 'warmup_ratio': 0.11987156815341725}. Best is trial 7 with value: 41.8359.


max_steps is given, it will override any value given in num_train_epochs


Step,Training Loss,Validation Loss


Trial   9 | r=128 α=64 drop=0.05 lr=6.39e-04 ep=5 bs=16 | ROUGE-1=39.4863
[I 2026-04-12 07:58:51,842] Trial 9 finished with value: 39.4863 and parameters: {'r': 128, 'lora_alpha': 64, 'lora_dropout': 0.04630922856909668, 'num_train_epochs': 5, 'learning_rate': 0.0006389963681824474, 'batch_size': 16, 'warmup_ratio': 0.11158690595251297}. Best is trial 7 with value: 41.8359.

════════════════════════════════════════════════════════════
🏆 Best Trial:
   ROUGE-1 : 41.8359
   r                     : 128
   lora_alpha            : 64
   lora_dropout          : 0.0339196572483829
   num_train_epochs      : 3
   learning_rate         : 0.0005578403009782301
   batch_size            : 32
   warmup_ratio          : 0.13567533266935894

📊 All trial results saved → ./optuna_46gb/optuna_trials.csv


In [ ]:

## 📋 Cell 9 — Train Best LoRA Model with Optimal Parameters

best_params = study.best_trial.params
print("Best params:", json.dumps(best_params, indent=2))

best_base = AutoModelForSeq2SeqLM.from_pretrained(
    MODEL_NAME, torch_dtype=torch.bfloat16                   # (GPU-friendly dtype)
).to(DEVICE)

best_cfg = LoraConfig(
    task_type=TaskType.SEQ_2_SEQ_LM,
    r=best_params["r"],
    lora_alpha=best_params["lora_alpha"],
    lora_dropout=best_params["lora_dropout"],
    target_modules=LORA_TARGET_MODULES,
    bias="none",
)
best_peft_model = get_peft_model(best_base, best_cfg)

trainable, total, pct = count_params(best_peft_model)
print(f"\nTrainable : {trainable:,}  ({pct:.2f}% of {total:,} total)")
print(f"Param reduction vs full fine-tune: {(1-pct/100)*100:.1f}%")

best_args = TrainingArguments(
    output_dir=LORA_BEST_DIR,
    num_train_epochs=best_params["num_train_epochs"],
    max_steps=100,                    # Raise to 300-500 for a longer full run
    per_device_train_batch_size=best_params["batch_size"],
    per_device_eval_batch_size=best_params["batch_size"],
    learning_rate=best_params["learning_rate"],
    warmup_ratio=best_params["warmup_ratio"],
    weight_decay=0.01,
    lr_scheduler_type="cosine",
    evaluation_strategy="steps",
    eval_steps=10,
    save_strategy="steps",
    save_steps=10,
    save_total_limit=3,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    fp16=False,
     
    bf16=torch.cuda.is_bf16_supported(),
    dataloader_num_workers=4,
    logging_steps=10,
    report_to="none",
)
best_trainer = Trainer(
    model=best_peft_model,
    args=best_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["validation"],
    callbacks=[EarlyStoppingCallback(early_stopping_patience=3)],
)
print("\n🔄 Training best LoRA model...")
best_trainer.train()
print("✅ Training complete.")






✅ Training complete.


In [19]:
# ── Save LoRA adapter weights + tokenizer ────────────────────────────────────
# Saves only the LoRA delta (~few MB) — base model loaded separately at inference.
best_peft_model.save_pretrained(LORA_BEST_DIR)
tokenizer.save_pretrained(LORA_BEST_DIR)

with open(os.path.join(LORA_BEST_DIR, "best_hyperparams.json"), "w") as f:
    json.dump(best_params, f, indent=2)

print(f"✅ LoRA adapter saved → {LORA_BEST_DIR}")
print(f"   Files: {os.listdir(LORA_BEST_DIR)}")

✅ LoRA adapter saved → ./output/lora_best_model
   Files: ['checkpoint-80', 'special_tokens_map.json', 'tokenizer_config.json', 'README.md', 'adapter_config.json', 'adapter_model.safetensors', 'checkpoint-100', 'checkpoint-90', 'spiece.model', 'tokenizer.json', 'best_hyperparams.json']


In [20]:
## 📋 Cell 10 — Qualitative Evaluation (Human-Readable)

results = []
for i in range(10):
    dialogue = dataset["test"][i]["dialogue"]
    prompt   = f"Summarize the following conversation.\n\n{dialogue}\n\nSummary: "
    results.append({
        "human_baseline": dataset["test"][i]["summary"],
        "original_model": generate_summary(original_model,  prompt),
        "full_finetune":  generate_summary(full_ft_model,   prompt),
        "lora_best":      generate_summary(best_peft_model, prompt),
    })

df_results = pd.DataFrame(results)
df_results.to_csv(os.path.join(BASE_DIR, "qualitative_comparison.csv"), index=False)
print(df_results[["human_baseline", "lora_best"]].head(5).to_string())


                                                                                                                                                                                                    human_baseline                                                                          lora_best
0                                              Ms. Dawson helps #Person1# to write a memo to inform every employee that they have to change the communication method and should not use Instant Messaging anymore.                        #Person1# asks #Person2# to take a dictation for #Person1#.
1  In order to prevent employees from wasting time on Instant Message programs, #Person1# decides to terminate the use of those programs and asks Ms. Dawson to send out a memo to all employees by the afternoon.                        #Person1# asks #Person2# to take a dictation for #Person1#.
2                                  Ms. Dawson takes a dictation for #Person1# about prohibiting the use of Instant Mes

In [21]:

## 📋 Cell 11 — Quantitative ROUGE Comparison


print("📊 Computing ROUGE on test set (100 samples each)...")

scores_zero = compute_rouge(original_model,  tokenizer, tokenized_datasets["test"], 100)
scores_full = compute_rouge(full_ft_model,   tokenizer, tokenized_datasets["test"], 100)
scores_lora = compute_rouge(best_peft_model, tokenizer, tokenized_datasets["test"], 100)

metrics = pd.DataFrame({
    "Model":   ["Zero-Shot (Original)", "Full Fine-Tune", "LoRA Best (Optuna)"],
    "ROUGE-1": [scores_zero["rouge1"], scores_full["rouge1"], scores_lora["rouge1"]],
    "ROUGE-2": [scores_zero["rouge2"], scores_full["rouge2"], scores_lora["rouge2"]],
    "ROUGE-L": [scores_zero["rougeL"], scores_full["rougeL"], scores_lora["rougeL"]],
})
metrics.to_csv(os.path.join(BASE_DIR, "rouge_comparison.csv"), index=False)

print("\n" + "═" * 60)
print(metrics.to_string(index=False))
print("═" * 60)
delta_r1 = scores_lora["rouge1"] - scores_full["rouge1"]
delta_r2 = scores_lora["rouge2"] - scores_full["rouge2"]
print(f"\n📈 LoRA vs Full Fine-Tune Delta:")
print(f"   ROUGE-1: {delta_r1:+.4f}")
print(f"   ROUGE-2: {delta_r2:+.4f}")


📊 Computing ROUGE on test set (100 samples each)...



════════════════════════════════════════════════════════════
               Model  ROUGE-1  ROUGE-2  ROUGE-L
Zero-Shot (Original)  24.5539   6.6609  21.1013
      Full Fine-Tune  42.0962  15.1618  33.2184
  LoRA Best (Optuna)  30.4495  10.0011  26.7117
════════════════════════════════════════════════════════════

📈 LoRA vs Full Fine-Tune Delta:
   ROUGE-1: -11.6467
   ROUGE-2: -5.1607


In [23]:
## 📋 Cell 12 — Local Inference (Reload Saved LoRA Weights)


# Reload saved LoRA adapter — for local testing or web-app integration
base_for_inference = AutoModelForSeq2SeqLM.from_pretrained(
    MODEL_NAME, torch_dtype=torch.bfloat16
).to(DEVICE)

inference_model = PeftModel.from_pretrained(base_for_inference, LORA_BEST_DIR)
inference_model.eval()
tok_inf = AutoTokenizer.from_pretrained(LORA_BEST_DIR)
print(f"✅ LoRA model reloaded from {LORA_BEST_DIR}")

test_dialogue = dataset["test"][5]["dialogue"]
prompt = f"Summarize the following conversation.\n\n{test_dialogue}\n\nSummary: "
result = generate_summary(inference_model, prompt)

print(f"\nDialogue (snippet): {test_dialogue[:200]}...")
print(f"\nLoRA Summary  : {result}")
print(f"Human Summary : {dataset['test'][5]['summary']}")


✅ LoRA model reloaded from ./output/lora_best_model

Dialogue (snippet): #Person1#: You're finally here! What took so long?
#Person2#: I got stuck in traffic again. There was a terrible traffic jam near the Carrefour intersection.
#Person1#: It's always rather congested do...

LoRA Summary  : #Person1# thinks that #Person1# should try to find a different route to get home.
Human Summary : #Person2# complains to #Person1# about the traffic jam, #Person1# suggests quitting driving and taking public transportation instead.


In [24]:
## 📋 Cell 13 — Optuna Visualization

try:
    import optuna.visualization as vis
    vis.plot_optimization_history(study).write_html(
        os.path.join(OPTUNA_DIR, "optimization_history.html"))
    vis.plot_param_importances(study).write_html(
        os.path.join(OPTUNA_DIR, "param_importances.html"))
    vis.plot_parallel_coordinate(study).write_html(
        os.path.join(OPTUNA_DIR, "parallel_coordinate.html"))
    print("✅ Optuna interactive plots saved to:", OPTUNA_DIR)
except Exception as e:
    print(f"Plot skipped: {e}")

top5 = study.trials_dataframe().sort_values("value", ascending=False).head(5)
print("\nTop 5 Trials by ROUGE-1:")
print(top5.to_string(index=False))


Plot skipped: Tried to import 'plotly' but failed. Please make sure that the package is installed correctly to use this feature. Actual error: No module named 'plotly'.

Top 5 Trials by ROUGE-1:
 number   value             datetime_start          datetime_complete               duration  params_batch_size  params_learning_rate  params_lora_alpha  params_lora_dropout  params_num_train_epochs  params_r  params_warmup_ratio    state
      7 41.8359 2026-04-12 07:53:43.143821 2026-04-12 07:55:51.364235 0 days 00:02:08.220414                 32              0.000558                 64             0.033920                        3       128             0.135675 COMPLETE
      8 41.4780 2026-04-12 07:55:51.366595 2026-04-12 07:57:33.882983 0 days 00:01:42.516388                 24              0.000655                 64             0.040854                        3       128             0.119872 COMPLETE
      4 40.9068 2026-04-12 07:48:30.343166 2026-04-12 07:50:39.496357 0 days 00:02:09.15

In [25]:

## 📋 Cell 14 — Final Summary


print("=" * 60)
print("      LORA FINE-TUNE OPTIMIZATION — FINAL SUMMARY")
print("=" * 60)
print(f"\n📂 Output: {os.path.abspath(BASE_DIR)}")
print("\n📁 Saved Artifacts:")
for root, dirs, files in os.walk(BASE_DIR):
    depth = root.replace(BASE_DIR, "").count(os.sep)
    indent = "  " * depth
    print(f"{indent}📂 {os.path.basename(root)}/")
    for fname in files:
        sz = os.path.getsize(os.path.join(root, fname)) / 1e6
        print(f"{indent}  📄 {fname}  ({sz:.2f} MB)")

print("\n🔧 Best LoRA Hyperparameters:")
for k, v in best_params.items():
    print(f"  {k:<22}: {v}")

print("\n📊 ROUGE Scores:")
print(metrics.to_string(index=False))
print("\n✅ Done. Weights ready for local inference or web-app deployment.")


      LORA FINE-TUNE OPTIMIZATION — FINAL SUMMARY

📂 Output: /teamspace/studios/this_studio/Generative_AI_Projects/output

📁 Saved Artifacts:
📂 output/
  📄 qualitative_comparison.csv  (0.01 MB)
  📄 rouge_comparison.csv  (0.00 MB)
  📂 optuna_trials/
  📂 logs/
  📂 lora_best_model/
    📄 special_tokens_map.json  (0.00 MB)
    📄 tokenizer_config.json  (0.02 MB)
    📄 README.md  (0.01 MB)
    📄 adapter_config.json  (0.00 MB)
    📄 adapter_model.safetensors  (28.33 MB)
    📄 spiece.model  (0.79 MB)
    📄 tokenizer.json  (2.42 MB)
    📄 best_hyperparams.json  (0.00 MB)
    📂 checkpoint-80/
      📄 README.md  (0.01 MB)
      📄 scheduler.pt  (0.00 MB)
      📄 trainer_state.json  (0.00 MB)
      📄 adapter_config.json  (0.00 MB)
      📄 adapter_model.safetensors  (28.33 MB)
      📄 rng_state.pth  (0.01 MB)
      📄 optimizer.pt  (56.74 MB)
      📄 training_args.bin  (0.00 MB)
    📂 checkpoint-100/
      📄 README.md  (0.01 MB)
      📄 scheduler.pt  (0.00 MB)
      📄 trainer_state.json  (0.00 MB)
  

In [28]:
import os
import zipfile

# ── Paths ────────────────────────────────────────────────────────────────────
OUTPUT_DIR = "output"
MERGED_DIR = os.path.join(OUTPUT_DIR, "output_merged")
ZIP_PATH = os.path.join(OUTPUT_DIR, "output_merged.zip")

os.makedirs(MERGED_DIR, exist_ok=True)

# ── Merge LoRA into base model ───────────────────────────────────────────────
print("🔄 Merging LoRA adapter with base model...")
merged_model = best_peft_model.merge_and_unload()

# ── Save merged full model + tokenizer ───────────────────────────────────────
print(f"💾 Saving merged model to: {MERGED_DIR}")
merged_model.save_pretrained(MERGED_DIR)
tokenizer.save_pretrained(MERGED_DIR)

print("✅ Merged model saved successfully.")
print(f"📂 Files in merged folder: {os.listdir(MERGED_DIR)}")

# ── Create ZIP file ──────────────────────────────────────────────────────────
print(f"🗜️ Creating zip file: {ZIP_PATH}")
with zipfile.ZipFile(ZIP_PATH, "w", zipfile.ZIP_DEFLATED) as zipf:
    for root, dirs, files in os.walk(MERGED_DIR):
        for file in files:
            file_path = os.path.join(root, file)
            arcname = os.path.relpath(file_path, OUTPUT_DIR)
            zipf.write(file_path, arcname)

print("✅ ZIP file created successfully.")
print(f"📦 ZIP saved at: {ZIP_PATH}")

# ── Final summary ────────────────────────────────────────────────────────────
print("\n🎯 Done!")
print(f"1. Merged model folder: {MERGED_DIR}")
print(f"2. Zip file: {ZIP_PATH}")
print("3. Download the ZIP from Lightning.AI file browser.")

🔄 Merging LoRA adapter with base model...
💾 Saving merged model to: output/output_merged
✅ Merged model saved successfully.
📂 Files in merged folder: ['config.json', 'special_tokens_map.json', 'tokenizer_config.json', 'model.safetensors', 'generation_config.json', 'spiece.model', 'tokenizer.json']
🗜️ Creating zip file: output/output_merged.zip
✅ ZIP file created successfully.
📦 ZIP saved at: output/output_merged.zip

🎯 Done!
1. Merged model folder: output/output_merged
2. Zip file: output/output_merged.zip
3. Download the ZIP from Lightning.AI file browser.
